In [14]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
df = pd.read_csv('./data/02-processed/marvel_rivals_final.csv')

In [15]:
#Making temporary X and y just to have dataframes to stratify, unneccesary columns are not removed here
#X = df.drop( columns = ['series_categorical'] )
#y = df['series_categorical']
#groups = df['session_id']

#Making test and train dfs stratified by gameplay session
#gss = GroupShuffleSplit(n_splits = 1, train_size = 0.8, random_state = 42)
#train_idx, test_idx = next(gss.split(X, y, groups))
#train_df = df.iloc[train_idx]
#test_df = df.iloc[test_idx]

#making sure stratification worked
#print(f"Total sessions: {df['session_id'].nunique()}")
#print(f"Train sessions: {train_df['session_id'].nunique()}")
#print(f"Test sessions: {test_df['session_id'].nunique()}")

In [16]:
#Calculate each player's loss streak
df = df.sort_values(['anon_player_id', 'match_time'])
df['loss_streak'] = df.groupby('anon_player_id')['is_win'].transform(
    lambda x: x.shift().fillna(False).cumsum().where(~x.shift().fillna(True), 0)
)

C:\Users\Yang Hu\AppData\Local\Temp\ipykernel_34324\2108200177.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.shift().fillna(False).cumsum().where(~x.shift().fillna(True), 0)
C:\Users\Yang Hu\AppData\Local\Temp\ipykernel_34324\2108200177.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.shift().fillna(False).cumsum().where(~x.shift().fillna(True), 0)
C:\Users\Yang Hu\AppData\Local\Temp\ipykernel_34324\2108200177.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in 

In [17]:
#NANs in the performance_score column, about 6% of total matches, you choose what to do with them
#Preprocessing


numerical_features = ['match_duration', 'SR_change', 'performance_score', 'loss_streak']
categorical_features = ['is_win']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])
model = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000))])

In [19]:
#Training
X = df.dropna(subset = ['performance_score'])[numerical_features + categorical_features]
y = df.dropna(subset=['performance_score'])['series_categorical']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y)
model.fit(X_train, y_train)
print(X_train, X_test, y_train, y_test)

c:\Users\Yang Hu\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


       match_duration  SR_change  performance_score  loss_streak  is_win
36929      338.757079 -17.750970          -2.533409            0   False
17689      432.699518 -16.181676          -1.825973            0   False
45718     1018.859228  20.538193           1.157674            0    True
70976      687.703339  18.136629          -0.641838            0    True
43801     1000.583341 -22.241610          -0.430180           16   False
...               ...        ...                ...          ...     ...
69941      948.259059 -19.296538          -0.117507            5   False
52819      834.822875 -21.339385           0.823763            0   False
40305      508.221143  19.145105           0.028024            0    True
33632      637.092682 -22.849206          -3.564699            0   False
26683      547.205450  19.608171          -1.248460            0    True

[53392 rows x 5 columns]        match_duration  SR_change  performance_score  loss_streak  is_win
32946      857.210575  20